# 将脚本改造为 Notebook 版本

本 Notebook 将原始 PyTorch 脚本拆分为多个可独立执行的单元，便于调试、观察中间结果和分步训练。

## 1) 安装与启动 Jupyter 内核（VS Code）

首次使用可在此单元安装依赖；运行后在 VS Code 右上角选择 Python 内核（建议选择你的 DL 环境）。

In [3]:
# 如已安装可跳过
# %pip install -U pip
# %pip install ipykernel notebook torch torchvision pillow

import sys
print(sys.executable)
print("内核已就绪")

d:\anaconda\envs\DL\python.exe
内核已就绪


In [4]:
# 解决 Windows 下 OpenMP 重复加载导致的内核崩溃问题
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
print("KMP_DUPLICATE_LIB_OK set")

KMP_DUPLICATE_LIB_OK set


## 2) 导入依赖并检查 PyTorch 版本

导入训练与数据处理所需模块，并确认当前 PyTorch 版本。

In [5]:
import os
import re
import random
import torch
import pandas as pd
import torch.nn as nn
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader

print("torch version:", torch.__version__)

torch version: 2.8.0+cpu


## 3) 配置数据路径（基于当前工作目录）

Notebook 中没有 __file__，这里使用当前工作目录并手动拼接数据路径。

In [6]:
# 若你的 Notebook 不在该目录运行，可手动指定项目根目录
project_root = os.path.dirname(os.getcwd())   # 当前工作目录
# 例如：project_root = r"d:/CZ_WorkSpace/ai学习/深度学习实践"

train_dir = os.path.join(project_root, "MNIST_kaggle", "trainingSet")
test_dir = os.path.join(project_root, "MNIST_kaggle", "testSet")

print("cwd:", os.getcwd())
print("project_root:", project_root)
print("train_dir exists:", os.path.exists(train_dir), "train_dir:", train_dir)
print("test_dir exists:", os.path.exists(test_dir), "test_dir:", test_dir)

cwd: d:\CZ_WorkSpace\ai学习\深度学习实践\task1\task1_1
project_root: d:\CZ_WorkSpace\ai学习\深度学习实践\task1
train_dir exists: True train_dir: d:\CZ_WorkSpace\ai学习\深度学习实践\task1\MNIST_kaggle\trainingSet
test_dir exists: True test_dir: d:\CZ_WorkSpace\ai学习\深度学习实践\task1\MNIST_kaggle\testSet


## 4) 定义 MyDataset（有标签/无标签双模式）

保留原有数据集逻辑：灰度化、缩放、ToTensor、展平 784；支持训练集有标签和测试集无标签。

In [7]:
class MyDataset(Dataset):
    def __init__(self, data_dir, labeled=True):
        self.data_dir = data_dir
        self.labeled = labeled
        self.samples = []
        self.transform = transforms.Compose([
            transforms.Grayscale(num_output_channels=1),
            transforms.Resize((28, 28)),
            transforms.ToTensor(),
        ])

        if self.labeled:
            class_names = [d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))]
            class_names = sorted(class_names, key=lambda x: int(x) if x.isdigit() else x)
            self.class_to_idx = {name: int(name) for name in class_names}
            for class_name in class_names:
                class_dir = os.path.join(data_dir, class_name)
                for file_name in os.listdir(class_dir):
                    if file_name.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
                        file_path = os.path.join(class_dir, file_name)
                        label = self.class_to_idx[class_name]
                        self.samples.append((file_path, label))
        else:
            self.class_to_idx = {}
            for file_name in os.listdir(data_dir):
                file_path = os.path.join(data_dir, file_name)
                if os.path.isfile(file_path) and file_name.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
                    self.samples.append(file_path)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        if self.labeled:
            file_path, label = self.samples[idx]
        else:
            file_path = self.samples[idx]
            label = None

        image = Image.open(file_path)
        image = self.transform(image)
        image = image.view(-1)  # [1, 28, 28] -> [784]

        if self.labeled:
            return image, torch.tensor(label, dtype=torch.long)
        return image, file_path

## 5) 构建 DataLoader 并检查 batch 形状

训练集使用 labeled=True，测试集按你当前结构使用 labeled=False。

In [8]:
mydataset = MyDataset(data_dir=train_dir, labeled=True)
mydataset_loader = DataLoader(mydataset, batch_size=32, shuffle=True)

test_dataset = MyDataset(data_dir=test_dir, labeled=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Train samples: {len(mydataset)}")
print(f"Test samples: {len(test_dataset)}")

for batch_idx, (images, labels) in enumerate(mydataset_loader):
    print(f"Batch {batch_idx}: images={images.shape}, labels={labels.shape}")
    print("Label sample:", labels[:10].tolist())
    if batch_idx == 1:
        break

Train samples: 42000
Test samples: 28000
Batch 0: images=torch.Size([32, 784]), labels=torch.Size([32])
Label sample: [2, 1, 7, 3, 6, 5, 4, 8, 5, 3]
Batch 1: images=torch.Size([32, 784]), labels=torch.Size([32])
Label sample: [7, 7, 6, 7, 5, 7, 4, 1, 2, 3]


## 6) 定义全连接网络 MyNetwork

网络结构为 784 -> 128 -> 10，输出 logits。

In [9]:
class MyNetwork(nn.Module):
    def __init__(self):
        super(MyNetwork, self).__init__()
        self.layer1 = nn.Linear(784, 128)
        self.act1 = nn.ReLU()
        self.layer2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.act1(self.layer1(x))
        x = self.layer2(x)
        return x

## 7) 定义评估函数 evaluate 与单样本推理 single_sample_check

测试集无标签时会跳过准确率；单样本推理输出预测类别与置信度。

In [10]:
def evaluate(model, data_loader):
    if len(data_loader.dataset) == 0 or not data_loader.dataset.labeled:
        print("Warning: test set has no labels, skip accuracy evaluation.")
        return None

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in data_loader:
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    model.train()
    if total == 0:
        print("Warning: test set is empty, skip accuracy evaluation.")
        return None
    return correct / total


def single_sample_check(model, dataset):
    if len(dataset) == 0:
        print("Single image validation skipped: test dataset is empty.")
        return

    sample_idx = random.randint(0, len(dataset) - 1)
    sample = dataset[sample_idx]
    if dataset.labeled:
        image, label = sample
        image_path, _ = dataset.samples[sample_idx]
    else:
        image, image_path = sample
        label = None

    model.eval()
    with torch.no_grad():
        logits = model(image.unsqueeze(0))
        probs = torch.softmax(logits, dim=1)
        pred = torch.argmax(probs, dim=1).item()
        confidence = probs[0, pred].item()
    model.train()

    print("\nSingle Image Validation")
    print(f"Image: {image_path}")
    if label is not None:
        print(f"True Label: {label.item()}, Pred Label: {pred}, Confidence: {confidence:.4f}")
    else:
        print(f"Pred Label: {pred}, Confidence: {confidence:.4f} (no ground-truth label)")

## 8) 定义损失函数与优化器

使用 CrossEntropyLoss 与 Adam(lr=0.001)，并做一次随机输入可用性检查。

In [11]:
loss_fn = nn.CrossEntropyLoss()
out = MyNetwork()
optimizer = torch.optim.Adam(out.parameters(), lr=0.001)

# 快速可用性检查
_ = loss_fn(out(torch.randn(32, 784)), torch.randint(0, 10, (32,)))
print("model check ok")

model check ok


## 9) 编写训练循环（含测试评估与模型保存）

每个 epoch 执行训练；若测试集有标签则评估准确率；每 2 个 epoch 保存一次模型。

In [12]:
epochs = 10
os.makedirs("checkpoints", exist_ok=True)

epoch_losses = []
epoch_accs = []

for epoch in range(epochs):
    out.train()
    running_loss = 0.0
    sample_count = 0
    for batch_idx, (images, labels) in enumerate(mydataset_loader):
        output = out(images)
        loss = loss_fn(output, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        running_loss += loss.item() * batch_size
        sample_count += batch_size

        print(f"Epoch {epoch}, Batch {batch_idx}, Loss: {loss.item():.6f}")

    epoch_loss = running_loss / max(sample_count, 1)
    epoch_losses.append(epoch_loss)

    test_acc = evaluate(out, test_loader)
    if test_acc is not None:
        epoch_accs.append(test_acc)
        print(f"Epoch {epoch}, Test Accuracy: {test_acc * 100:.2f}%")
    else:
        epoch_accs.append(None)

    if epoch % 1 == 0:
        ckpt_path = os.path.join("checkpoints", f"model_epoch_{epoch}.pth")
        torch.save(out.state_dict(), ckpt_path)
        print("Saved:", ckpt_path)

Epoch 0, Batch 0, Loss: 2.296938
Epoch 0, Batch 1, Loss: 2.310750
Epoch 0, Batch 2, Loss: 2.219379
Epoch 0, Batch 3, Loss: 2.227668
Epoch 0, Batch 4, Loss: 2.203538
Epoch 0, Batch 5, Loss: 2.102791
Epoch 0, Batch 6, Loss: 2.085211
Epoch 0, Batch 7, Loss: 2.135899
Epoch 0, Batch 8, Loss: 2.028502
Epoch 0, Batch 9, Loss: 2.066352
Epoch 0, Batch 10, Loss: 1.953009
Epoch 0, Batch 11, Loss: 1.974849
Epoch 0, Batch 12, Loss: 2.029665
Epoch 0, Batch 13, Loss: 1.806929
Epoch 0, Batch 14, Loss: 1.816746
Epoch 0, Batch 15, Loss: 1.935220
Epoch 0, Batch 16, Loss: 1.768692
Epoch 0, Batch 17, Loss: 1.552198
Epoch 0, Batch 18, Loss: 1.680870
Epoch 0, Batch 19, Loss: 1.519499
Epoch 0, Batch 20, Loss: 1.609265
Epoch 0, Batch 21, Loss: 1.581501
Epoch 0, Batch 22, Loss: 1.748310
Epoch 0, Batch 23, Loss: 1.554196
Epoch 0, Batch 24, Loss: 1.368347
Epoch 0, Batch 25, Loss: 1.494142
Epoch 0, Batch 26, Loss: 1.409348
Epoch 0, Batch 27, Loss: 1.241268
Epoch 0, Batch 28, Loss: 1.314222
Epoch 0, Batch 29, Loss:

In [3]:
# 9.1) 训练曲线可视化
epochs_axis = list(range(1, len(epoch_losses) + 1))

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(epochs_axis, epoch_losses, marker="o")
plt.title("Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True, linestyle="--", alpha=0.4)

plt.subplot(1, 2, 2)
valid_acc = [a for a in epoch_accs if a is not None]
valid_epochs = [i + 1 for i, a in enumerate(epoch_accs) if a is not None]
if valid_acc:
    plt.plot(valid_epochs, valid_acc, marker="o", color="green")
    plt.ylim(0, 1)
plt.title("Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

NameError: name 'epoch_losses' is not defined

## 10) 训练后单图验证与可选模型加载推理

先做一次单图验证；再演示如何加载保存的权重做推理。

In [14]:
# 训练后单图验证
single_sample_check(out, test_dataset)

# 可选：加载某个 checkpoint 后再做单图推理
# model2 = MyNetwork()
# model2.load_state_dict(torch.load(os.path.join("checkpoints", "model_epoch_8.pth"), map_location="cpu"))
# single_sample_check(model2, test_dataset)


Single Image Validation
Image: d:\CZ_WorkSpace\ai学习\深度学习实践\task1\MNIST_kaggle\testSet\img_23881.jpg
Pred Label: 2, Confidence: 1.0000 (no ground-truth label)


In [17]:
# 11) 加载模型参数并对测试集推理（导出 Kaggle 提交文件）
import pandas as pd
from torch.utils.data import DataLoader

# TODO: 修改为你要加载的权重文件路径
ckpt_path = os.path.join("checkpoints", "model_epoch_8.pth")

infer_model = MyNetwork()
infer_model.load_state_dict(torch.load(ckpt_path, map_location="cpu"))
infer_model.eval()

# 测试集无标签，推理时 batch 处理
infer_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

pred_records = []
with torch.no_grad():
    for images, file_paths in infer_loader:
        logits = infer_model(images)
        preds = torch.argmax(logits, dim=1).cpu().numpy().tolist()
        for path, pred in zip(file_paths, preds):
            base = os.path.basename(path)
            match = re.search(r"\d+", base)
            if match is None:
                raise ValueError(f"Cannot parse image id from filename: {base}")
            image_id = int(match.group())
            pred_records.append((image_id, pred))

# 按 ImageId 数值升序，符合 Kaggle 提交要求
pred_records.sort(key=lambda x: x[0])
submit_df = pd.DataFrame(pred_records, columns=["ImageId", "Label"])

# 保存为 CSV，方便提交到 Kaggle
submit_path = "kaggle_submit.csv"
submit_df.to_csv(submit_path, index=False)
print(f"Saved submission file: {submit_path}")
print(submit_df.head())

Saved submission file: kaggle_submit.csv
   ImageId  Label
0        1      2
1        2      0
2        3      9
3        4      9
4        5      3
